# libgltf

Minecraft 26.3 的 glTF 渲染层。负责模型加载、材质、动画、LOD，以及 Vulkan / OpenGL 双后端的 GPU 绘制。
代码集中在 `src/main/kotlin/com/micheanl/libgltf`。

## 一帧的绘制路径

```
GltfSceneRenderer.submit        收集实例，按材质分组提交
        │
GpuSubmitRenderer               按 GpuBatchKey 分组准备
        │
GpuBatch                        写实例缓冲，选择绘制路径
        │
MeshletDispatcher               mesh：准备 meshlet 状态；间接：生成命令
        │
VulkanGpuDriver / GlGpuDriver   mesh 管线缓存与分发
        │
mesh shader                     剔除 → 紧凑顶点解码 → 发射
        │
gpu_mesh.fsh                    与 Minecraft 材质管线一致的片元阶段
```

支持 mesh shader 的设备默认走 mesh 路径；没有扩展的设备降级到计算剔除 + 间接绘制。

## 模块地图

| 模块 | 内容 |
|---|---|
| `asset` / `model` | glTF 解析与数据结构 |
| `material` / `texture` | 材质扩展、贴图生成与绑定 |
| `animation` / `lod` | 动画状态机、LOD 策略与简化 |
| `render.gpu` | GPU 资源、meshlet 存储、能力探测 |
| `render.vulkan` / `render.gl` | 两套后端的 mesh 与间接绘制 |
| `render.feature` | 提交分组、批处理、FeatureRenderer 接入 |
| `integration` | 方块、实体、物品渲染集成 |

In [ ]:
import java.nio.file.Files
import java.nio.file.Paths

val root = Paths.get(System.getProperty("user.dir"), "src/main/kotlin/com/micheanl/libgltf")
Files.walk(root).use { paths ->
    paths.filter(Files::isDirectory).filter { it != root }.sorted().forEach { dir ->
        val count = Files.list(dir).use { it.count() }
        println("%-42s %d 个文件".format(root.relativize(dir).toString(), count))
    }
}

## 值得先读的文件

- `MeshletStorage.kt` — meshlet 的 GPU 数据布局，每个顶点 16 字节
- `MeshletDispatcher.kt` — 每帧分发、描述符跨帧缓存
- `VulkanNvMeshPipelineCache.kt` — NV mesh 管线、批量与剔除宏
- `RenderConfig.kt` — 性能开关与 config 文件映射
- `GltfSceneRenderer.kt` — 提交入口，场景到 GPU 的第一站

In [ ]:
val files = listOf(
    "render/GltfSceneRenderer.kt",
    "render/feature/GpuBatch.kt",
    "render/vulkan/MeshletDispatcher.kt",
    "render/vulkan/VulkanGpuDriver.kt",
    "render/gpu/MeshletStorage.kt",
    "render/vulkan/RenderConfig.kt"
)
files.forEach { rel ->
    println("===== $rel =====")
    Files.readAllLines(root.resolve(rel)).take(16).forEach(::println)
    println()
}

## 设计说明

顶点按 meshlet 单独打包：量化 16 位位置、oct16 法线、fp16 UV、8 位颜色，共 16 字节。mesh shader 顺序读取，避免全局顶点缓冲的随机访问和重复顶点带宽。

NV 路径每个工作组处理 4 个 meshlet，把组数压到原来的四分之一；剔除分三级——球体视锥、紧致锥背面、上一帧深度遮挡。深度遮挡在设备深度空间直接比较，不依赖线性深度换算约定。

存储描述符集按缓冲区身份跨帧缓存，稳态下每帧零更新。

双后端差异只有一处：Vulkan 用 64/64 meshlet 配合批量，OpenGL 用 256/256 减少任务组。其余路径共用同一套紧凑数据与剔除逻辑。